In [0]:
#DATA TRANSFORMATION BOOKINGS TABLE

CATALOG = "airbnb_obs"
from pyspark.sql import functions as F

bronze = spark.table(f"{CATALOG}.bronze.bookings")

#  CAST: try_cast returns NULL on a bad value instead of throwing
typed = (bronze
    .withColumn("listing_id",     F.expr("try_cast(listing_id AS INT)"))
    .withColumn("booking_date",   F.expr("try_cast(booking_date AS DATE)"))
    .withColumn("nights_booked",  F.expr("try_cast(nights_booked AS INT)"))
    .withColumn("booking_amount", F.expr("try_cast(booking_amount AS DOUBLE)"))
    .withColumn("cleaning_fee",   F.expr("try_cast(cleaning_fee AS DOUBLE)"))
    .withColumn("service_fee",    F.expr("try_cast(service_fee AS DOUBLE)"))
    .withColumn("created_at",     F.expr("try_cast(created_at AS TIMESTAMP)"))
    .withColumn("_ingested_at_utc", F.expr("try_cast(_ingested_at_utc AS TIMESTAMP)"))
    # CLEAN: standardize the status text (trim spaces, lowercase) 
    .withColumn("booking_status", F.lower(F.trim(F.col("booking_status"))))
)

#  QUALITY GATE: what a valid Silver row must satisfy 
valid_condition = (
      F.col("listing_id").isNotNull()
    & F.col("nights_booked").between(1, 30)
    & F.col("booking_status").isin("confirmed", "cancelled")
)

# null-safe: a NULL condition (e.g. a failed cast) counts as INVALID, never lost
is_valid = F.coalesce(valid_condition, F.lit(False))

clean      = typed.filter(is_valid)
quarantine = typed.filter(~is_valid)

# SAVE CLEANED TABLE
(clean.write.mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable(f"{CATALOG}.silver.bookings"))

(quarantine.write.mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable(f"{CATALOG}.silver.bookings_quarantine"))

print(f"clean:       {clean.count()} rows -> silver.bookings")
print(f"quarantined: {quarantine.count()} rows -> silver.bookings_quarantine")

In [0]:
%sql

DESCRIBE airbnb_obs.silver.bookings;

In [0]:
#DATA TRANSFORMATION HOSTS TABLE

CATALOG = "airbnb_obs"
from pyspark.sql import functions as F

bronze_h = spark.table(f"{CATALOG}.bronze.hosts")

typed_h = (bronze_h
    .withColumn("host_id",        F.expr("try_cast(host_id AS INT)"))
    .withColumn("host_since",     F.expr("try_cast(host_since AS DATE)"))
    .withColumn("response_rate",  F.expr("try_cast(response_rate AS DOUBLE)"))
    .withColumn("created_at",     F.expr("try_cast(created_at AS TIMESTAMP)"))
    .withColumn("is_superhost",   F.lower(F.trim(F.col("is_superhost"))))
)

#QUALITY GATE
valid_h = (F.col("host_id").isNotNull()
         & F.col("host_since").isNotNull())
is_valid_h = F.coalesce(valid_h, F.lit(False))

# SAVE CLEANED TABLE
(typed_h.filter(is_valid_h).write.mode("overwrite").option("overwriteSchema","true")
        .saveAsTable(f"{CATALOG}.silver.hosts"))
(typed_h.filter(~is_valid_h).write.mode("overwrite").option("overwriteSchema","true")
        .saveAsTable(f"{CATALOG}.silver.hosts_quarantine"))

print(f"hosts clean: {typed_h.filter(is_valid_h).count()}, "
      f"quarantined: {typed_h.filter(~is_valid_h).count()}")

In [0]:
%sql  DESCRIBE airbnb_obs.silver.hosts;

SELECT * FROM airbnb_obs.silver.hosts LIMIT 10;

In [0]:
#DATA TRANSFORMATION LISTINGS TABLE

bronze_l = spark.table(f"{CATALOG}.bronze.listings")

typed_l = (bronze_l
    .withColumn("listing_id",      F.expr("try_cast(listing_id AS INT)"))
    .withColumn("host_id",         F.expr("try_cast(host_id AS INT)"))
    .withColumn("accommodates",    F.expr("try_cast(accommodates AS INT)"))
    .withColumn("bedrooms",        F.expr("try_cast(bedrooms AS INT)"))
    .withColumn("bathrooms",       F.expr("try_cast(bathrooms AS DOUBLE)"))
    .withColumn("price_per_night", F.expr("try_cast(price_per_night AS DOUBLE)"))
    .withColumn("created_at",      F.expr("try_cast(created_at AS TIMESTAMP)"))
    .withColumn("room_type",       F.lower(F.trim(F.col("room_type"))))
)

valid_l = (F.col("listing_id").isNotNull()
         & F.col("host_id").isNotNull()
         & (F.col("price_per_night") > 0)
         & F.col("accommodates").between(1, 16))
is_valid_l = F.coalesce(valid_l, F.lit(False))

(typed_l.filter(is_valid_l).write.mode("overwrite").option("overwriteSchema","true")
        .saveAsTable(f"{CATALOG}.silver.listings"))
(typed_l.filter(~is_valid_l).write.mode("overwrite").option("overwriteSchema","true")
        .saveAsTable(f"{CATALOG}.silver.listings_quarantine"))

print(f"listings clean: {typed_l.filter(is_valid_l).count()}, "
      f"quarantined: {typed_l.filter(~is_valid_l).count()}")

In [0]:
%sql  DESCRIBE airbnb_obs.silver.listings;

SELECT * FROM airbnb_obs.silver.listings LIMIT 10;

In [0]:
# ---------- LOG SILVER INTEGRITY VERDICT TO dq_results ----------

from datetime import datetime, timezone
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType, LongType
)

def log_silver_verdict(table_name, clean_df, quarantine_df, run_id):
    quarantined = quarantine_df.count()
    total       = clean_df.count() + quarantined
    status      = "PASS" if quarantined == 0 else "FAIL"

    schema = StructType([
        StructField("run_id",         StringType(),    True),
        StructField("check_ts",       TimestampType(), True),
        StructField("layer",          StringType(),    True),
        StructField("table_name",     StringType(),    True),
        StructField("column_name",    StringType(),    True),
        StructField("rule_name",      StringType(),    True),
        StructField("rule_type",      StringType(),    True),
        StructField("failed_records", LongType(),      True),
        StructField("total_records",  LongType(),      True),
        StructField("status",         StringType(),    True),
        StructField("details",        StringType(),    True),
    ])

    verdict = spark.createDataFrame(
        [(
            run_id,
            datetime.now(timezone.utc),
            "silver",
            table_name,
            None,
            "silver_quality_gate",
            "quality_gate",
            quarantined,
            total,
            status,
            None,
        )],
        schema
    )
    (verdict.write.mode("append").saveAsTable(f"{CATALOG}.monitoring.dq_results"))
    print(f"logged silver verdict: {table_name} -> {status}, {quarantined} quarantined")

# one shared run_id for this whole Silver run
run_id = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")

log_silver_verdict("bookings", clean,   quarantine,   run_id)
log_silver_verdict("hosts",    typed_h.filter(is_valid_h),  typed_h.filter(~is_valid_h),  run_id)
log_silver_verdict("listings", typed_l.filter(is_valid_l),  typed_l.filter(~is_valid_l),  run_id)

In [0]:
%sql -- TO READ THE DATA (LATEST VERDICT FOR EACH CHECK, PASSED OR FAILED)
CREATE OR REPLACE VIEW airbnb_obs.monitoring.v_current_dq_state AS
WITH ranked AS (
    SELECT
        layer,
        table_name,
        rule_name,
        status,
        failed_records,
        total_records,
        check_ts,
        run_id,
        ROW_NUMBER() OVER (
            PARTITION BY layer, table_name, rule_name
            ORDER BY check_ts DESC
        ) AS rn
    FROM airbnb_obs.monitoring.dq_results
)
SELECT
    layer,
    table_name,
    rule_name,
    status,
    failed_records,
    total_records,
    check_ts
FROM ranked
WHERE rn = 1;

In [0]:
%sql -- CHECK WHAT IS IN THE VIEW
SELECT layer, table_name, rule_name, status
FROM airbnb_obs.monitoring.v_current_dq_state
WHERE layer = 'silver';